## Program 2C

The full CCD workflow from programs 2A and 2B, with streamlined function calls and output.

In [3]:
# Import Libraries

import pandas as pd
import numpy as np
import time
from time import perf_counter_ns
from collections import defaultdict
from pathlib import Path
import re
import math
import gc

In [4]:
n_rows = 2000 #296168

In [5]:
# User Values

h_0 = 0.75
h_1 = 20
r_1 = 20
h_2 = 10
r_2 = 20

epsilon = 0.6 # max sweep volumetric over-representation (%)
beta = 1 # max angular deviation from SLERP (degrees)
delta_step = 1 # segmentation step angle resolution (degrees)

folder_path = Path(r"C:\S3_3DP_MotionPlanning\DataSet\Sorce\bunnyHead\waypoint")

EPS = 1e-12
TOL = 1e-12

In [6]:
# Data Import Function and Execution

def build_A_and_A_ref(
    folder_path=folder_path, n_rows=n_rows, h_0=h_0, h_1=h_1, r_1=r_1, h_2=h_2, r_2=r_2, epsilon=epsilon, beta=beta, delta_step=delta_step,
):

    if n_rows < 2:
        raise SystemExit("CCD analysis cannot be performed on less than 2 points.")
    
    A_COLS = {
        "x": 0, "y": 1, "z": 2,
        "x+": 3, "y+": 4, "z+": 5,
        "i3": 6, "j3": 7, "k3": 8,
        "h3": 9, "h4": 10, "r3": 11, "r4": 12,
        "q1": 13, "q2": 14, "q3": 15,
        "delta_seg": 16,
    }
    
    A_REF_COLS = {
        "x": 0, "y": 1, "z": 2,
        "i": 3, "j": 4, "k": 5,
        "layer_end": 6,
    }
   
    if r_2 < r_1:
        raise SystemExit("Invalid C2 geometry: expected r_2 >= r_1.")

    if epsilon <= 0.001:
        raise SystemExit("Abnormally low epsilon value, consider a value above 0.001.")

    if n_rows <= 0:
        raise SystemExit("n_rows must be greater than zero.")

    if delta_step <= 0:
        raise SystemExit("delta_step must be greater than zero.")

    h_delta_13_min = 0.8

    alpha_a = r_1 / h_1
    alpha_b = h_2 / h_1
    alpha_c = r_2 / h_1

    if not (0.2 <= alpha_a <= 3.0):
        raise SystemExit(f"alpha_a = {alpha_a:.6f} is outside the valid range 0.2 <= alpha_a <= 3.0.")

    if not (0.2 <= alpha_b <= 6.0):
        raise SystemExit(f"alpha_b = {alpha_b:.6f} is outside the valid range 0.2 <= alpha_b <= 6.0.")

    if not (0.2 <= alpha_c <= 6.0):
        raise SystemExit(f"alpha_c = {alpha_c:.6f} is outside the valid range 0.2 <= alpha_c <= 6.0.")

    if alpha_c < alpha_a:
        raise SystemExit("Invalid C2 geometry: expected alpha_c >= alpha_a, equivalent to r_2 >= r_1.")

    delta_max_a = 2.0 * math.atan(alpha_c / (1.0 + alpha_b))

    delta_max_b = 2.0 * (
        math.acos(h_delta_13_min / math.sqrt(1.0 + alpha_c**2))
        - math.atan(alpha_c)
    )

    delta_max_ab = min(delta_max_a, delta_max_b)

    def volume_c2_delta(delta):
        base_volume = math.pi * ((alpha_a**2 / 3.0) + (alpha_c**2 * alpha_b))

        sweep_term = delta * (
            (2.0 / 3.0) * alpha_a
            + alpha_c * (2.0 * alpha_b + alpha_b**2)
            + (2.0 / 3.0) * alpha_c**3
        )

        return base_volume + sweep_term

    def volume_c3_delta(delta):
        c = math.cos(delta / 2.0)
        s = math.sin(delta / 2.0)

        denominator = c - alpha_a * s

        if abs(denominator) < 1e-12:
            return math.inf

        cylinder_term = (
            (alpha_c * c + (1.0 + alpha_b) * s)**2
            * (alpha_b * c + 2.0 * alpha_c * s)
        )

        cone_term = (
            ((alpha_a * c + s)**2 * (c - alpha_c * s)**3)
            / (3.0 * denominator**2)
        )

        return math.pi * (cylinder_term + cone_term)

    def epsilon_delta(delta):
        v_c2 = volume_c2_delta(delta)
        v_c3 = volume_c3_delta(delta)
        return (v_c3 - v_c2) / v_c2

    if epsilon == 0:
        delta_max_c = 0.0

    elif epsilon_delta(delta_max_ab) <= epsilon:
        delta_max_c = delta_max_ab

    else:
        lo = 0.0
        hi = delta_max_ab

        for _ in range(80):
            mid = 0.5 * (lo + hi)

            if epsilon_delta(mid) <= epsilon:
                lo = mid
            else:
                hi = mid

        delta_max_c = lo

    delta_max = min(delta_max_a, delta_max_b, delta_max_c)

    if delta_max <= 0:
        raise SystemExit("delta_max must be greater than zero.")

    beta_rad = np.radians(beta)
    delta_step_rad = np.radians(delta_step)

    def natural_sort_key(path):
        return [
            int(part) if part.isdigit() else part.lower()
            for part in re.split(r"(\d+)", path.name)
        ]

    def slerp_unit_vectors(u1, u2, t_values, omega):
        """
        SLERP between two unit vectors.
        """
        sin_omega = np.sin(omega)

        if abs(sin_omega) < 1e-12:
            u = (1.0 - t_values[:, None]) * u1 + t_values[:, None] * u2
        else:
            w1 = np.sin((1.0 - t_values) * omega) / sin_omega
            w2 = np.sin(t_values * omega) / sin_omega
            u = w1[:, None] * u1 + w2[:, None] * u2

        norms = np.linalg.norm(u, axis=1)

        if np.any(norms == 0):
            raise ValueError("A zero-length orientation vector was produced during SLERP.")

        return u / norms[:, None]

    folder_path = Path(folder_path)
    txt_files = sorted(folder_path.glob("*.txt"), key=natural_sort_key)

    if not txt_files:
        raise ValueError(f"No .txt files were found in: {folder_path}")

    all_data = []
    rows_imported = 0

    for txt_file in txt_files:

        if rows_imported >= n_rows:
            break

        data = np.loadtxt(txt_file, dtype=np.float64)
        data = np.atleast_2d(data)

        if data.shape[1] != 6:
            raise ValueError(
                f"Expected 6 columns in {txt_file.name}, but found {data.shape[1]}."
            )

        original_file_rows = data.shape[0]
        remaining_rows = n_rows - rows_imported
        rows_to_take = min(remaining_rows, original_file_rows)

        data = data[:rows_to_take]

        layer_end = np.zeros((rows_to_take, 1), dtype=np.float64)

        if rows_to_take == original_file_rows:
            layer_end[-1, 0] = 1.0

        all_data.append(np.hstack((data, layer_end)))
        rows_imported += rows_to_take

    if not all_data:
        raise ValueError("No data was imported.")

    A_ref = np.vstack(all_data).astype(np.float64, copy=False)

    # Mark the final imported line as an endpoint, even if n_rows cuts through a txt file.
    A_ref[-1, 6] = 1.0

    del all_data
    gc.collect()

    layer_end_mask = A_ref[:, 6] == 1.0
    segment_source_rows = np.flatnonzero(~layer_end_mask)

    if len(segment_source_rows) == 0:
        raise ValueError("No valid movement rows were produced. Check the waypoint files and n_rows.")

    start_rows = A_ref[segment_source_rows, :6]
    end_rows = A_ref[segment_source_rows + 1, :6]

    dot_product = np.sum(start_rows[:, 3:6] * end_rows[:, 3:6], axis=1)
    dot_product = np.clip(dot_product, -1.0, 1.0)
    omega_values = np.arccos(dot_product)

    # Initial A columns:
    # x,y,z,x+,y+,z+,i,j,k,i+,j+,k+,q_1,q_2,omega
    A0 = np.empty((len(segment_source_rows), 15), dtype=np.float64)
    A0[:, 0:3] = start_rows[:, 0:3]
    A0[:, 3:6] = end_rows[:, 0:3]
    A0[:, 6:9] = start_rows[:, 3:6]
    A0[:, 9:12] = end_rows[:, 3:6]
    A0[:, 12] = segment_source_rows.astype(np.float64)
    A0[:, 13] = 0.0
    A0[:, 14] = omega_values

    # Primary segmentation

    primary_blocks = []

    for row in A0:

        omega = float(row[14])

        if (not np.isfinite(omega)) or omega <= 2.0 * beta_rad:
            out = row.copy()
            out[13] = 0.0
            primary_blocks.append(out[None, :])
            continue

        N_p = int(np.ceil((omega - 2.0 * beta_rad) / delta_step_rad))

        if N_p < 1:
            N_p = 1

        T_1 = row[0:3]
        T_2 = row[3:6]

        u_1 = row[6:9].astype(np.float64, copy=True)
        u_2 = row[9:12].astype(np.float64, copy=True)

        u_1_norm = np.linalg.norm(u_1)
        u_2_norm = np.linalg.norm(u_2)

        if u_1_norm == 0 or u_2_norm == 0:
            raise ValueError(f"Zero-length unit vector found at q_1 = {row[12]:.0f}.")

        u_1 = u_1 / u_1_norm
        u_2 = u_2 / u_2_norm

        theta_values = np.empty(N_p + 1, dtype=np.float64)
        theta_values[0] = 0.0
        theta_values[-1] = omega

        if N_p > 1:
            m_values = np.arange(1, N_p, dtype=np.float64)
            theta_values[1:-1] = beta_rad + m_values * delta_step_rad

        t_values = theta_values / omega

        T_values = T_1 + t_values[:, None] * (T_2 - T_1)
        u_values = slerp_unit_vectors(u_1, u_2, t_values, omega)

        block = np.empty((N_p, 15), dtype=np.float64)

        block[:, 0:3] = T_values[:-1]
        block[:, 3:6] = T_values[1:]
        block[:, 6:9] = u_values[:-1]
        block[:, 9:12] = u_values[1:]
        block[:, 12] = row[12]
        block[:, 13] = np.arange(N_p, dtype=np.float64)
        block[:, 14] = np.diff(theta_values)

        primary_blocks.append(block)

    A_primary = np.vstack(primary_blocks)

    del primary_blocks, A0
    gc.collect()

    # Secondary segmentation

    secondary_blocks = []

    for row in A_primary:

        delta_seg = float(row[14])

        if (not np.isfinite(delta_seg)) or delta_seg <= delta_max:
            out = np.empty(16, dtype=np.float64)
            out[0:14] = row[0:14]
            out[14] = 0.0
            out[15] = row[14]
            secondary_blocks.append(out[None, :])
            continue

        N_s = int(np.ceil(delta_seg / delta_max))

        if N_s < 1:
            N_s = 1

        new_delta_seg = delta_seg / N_s

        u_1 = row[6:9].astype(np.float64, copy=True)
        u_2 = row[9:12].astype(np.float64, copy=True)

        u_1_norm = np.linalg.norm(u_1)
        u_2_norm = np.linalg.norm(u_2)

        if u_1_norm == 0 or u_2_norm == 0:
            raise ValueError(
                f"Zero-length unit vector found at q_1 = {row[12]:.0f}, q_2 = {row[13]:.0f}."
            )

        u_1 = u_1 / u_1_norm
        u_2 = u_2 / u_2_norm

        t_values = np.linspace(0.0, 1.0, N_s + 1)
        u_values = slerp_unit_vectors(u_1, u_2, t_values, delta_seg)

        block = np.empty((N_s, 16), dtype=np.float64)

        # Positions remain unchanged during secondary segmentation.
        block[:, 0:6] = row[0:6]

        # Orientation is subdivided.
        block[:, 6:9] = u_values[:-1]
        block[:, 9:12] = u_values[1:]

        # q_1 and q_2 are maintained; q_3 is the secondary subdivision counter.
        block[:, 12] = row[12]
        block[:, 13] = row[13]
        block[:, 14] = np.arange(N_s, dtype=np.float64)
        block[:, 15] = new_delta_seg

        secondary_blocks.append(block)

    A_secondary = np.vstack(secondary_blocks)

    del secondary_blocks, A_primary
    gc.collect()

    # Establish u3

    u_1 = A_secondary[:, 6:9].astype(np.float64, copy=True)
    u_2 = A_secondary[:, 9:12].astype(np.float64, copy=True)

    u_1_norms = np.linalg.norm(u_1, axis=1)
    u_2_norms = np.linalg.norm(u_2, axis=1)

    if np.any(u_1_norms == 0):
        raise ValueError("At least one [i, j, k] vector has zero length.")

    if np.any(u_2_norms == 0):
        raise ValueError("At least one [i+, j+, k+] vector has zero length.")

    u_1 = u_1 / u_1_norms[:, None]
    u_2 = u_2 / u_2_norms[:, None]

    dot_product = np.sum(u_1 * u_2, axis=1)
    dot_product = np.clip(dot_product, -1.0, 1.0)

    omega_direct = np.arccos(dot_product)
    sin_omega = np.sin(omega_direct)

    u_3 = np.empty_like(u_1)

    small_angle_mask = np.abs(sin_omega) < 1e-12
    normal_mask = ~small_angle_mask

    u_3[small_angle_mask] = u_1[small_angle_mask]

    weight = np.sin(0.5 * omega_direct[normal_mask]) / sin_omega[normal_mask]

    u_3[normal_mask] = (
        weight[:, None] * u_1[normal_mask]
        + weight[:, None] * u_2[normal_mask]
    )

    u_3_norms = np.linalg.norm(u_3, axis=1)

    if np.any(u_3_norms == 0):
        raise ValueError("At least one midpoint vector [i3, j3, k3] has zero length.")

    u_3 = u_3 / u_3_norms[:, None]

    # Establish C3 geometry and final A

    delta = A_secondary[:, 15]
    half_delta = 0.5 * delta

    cos_half = np.cos(half_delta)
    sin_half = np.sin(half_delta)

    r3_denominator = h_1 * cos_half - r_1 * sin_half

    if np.any(np.abs(r3_denominator) < 1e-12):
        raise ValueError(
            "At least one r_3 denominator is close to zero. "
            "Check h_1, r_1, and delta_seg values."
        )

    h3 = h_1 * cos_half - r_2 * sin_half
    h4 = h_2 * cos_half + 2.0 * r_2 * sin_half

    r3 = (
        (r_1 * cos_half + h_1 * sin_half)
        *
        ((h_1 * cos_half - r_2 * sin_half) / r3_denominator)
    )

    r4 = r_2 * cos_half + (h_1 + h_2) * sin_half

    A = np.empty((A_secondary.shape[0], 17), dtype=np.float64)

    A[:, 0:6] = A_secondary[:, 0:6]
    A[:, 6:9] = u_3
    A[:, 9] = h3
    A[:, 10] = h4
    A[:, 11] = r3
    A[:, 12] = r4
    A[:, 13:16] = A_secondary[:, 12:15]
    A[:, 16] = A_secondary[:, 15]


    #print(f"alpha_a     = {alpha_a:.6f}")
    #print(f"alpha_b     = {alpha_b:.6f}")
    #print(f"alpha_c     = {alpha_c:.6f}")
    #print(f"delta_max   = {delta_max:.6f} rad ({math.degrees(delta_max):.6f} deg)")
    #print(f"2beta       = {np.radians(2 * beta):.6f} rad")
    #print(f"A_ref shape : {A_ref.shape}")
    #print(f"A shape     : {A.shape}")
    #print(f"Max delta_seg: {np.nanmax(A[:, 16]):.6f} rad ({np.degrees(np.nanmax(A[:, 16])):.6f} deg)")

    return A, A_ref, A_COLS, A_REF_COLS

#A, A_ref, A_COLS, A_REF_COLS = build_A_and_A_ref()

In [7]:
# PIT Helping Functions

def q1_to_ref_index(q1_value):

    if not np.isfinite(q1_value):
        return None

    idx = int(round(q1_value))
    return idx


def t_interval_for_axial_range(a0, du, s_min, s_max, eps=EPS):

    a0 = np.asarray(a0, dtype=float)
    n = a0.shape[0]

    t_lo = np.zeros(n, dtype=float)
    t_hi = np.ones(n, dtype=float)

    if s_max < s_min:
        return t_lo, t_hi, np.zeros(n, dtype=bool)

    if abs(du) <= eps:
        valid = (a0 >= s_min - eps) & (a0 <= s_max + eps)
        return t_lo, t_hi, valid

    t_a = (a0 - s_max) / du
    t_b = (a0 - s_min) / du

    lo = np.minimum(t_a, t_b)
    hi = np.maximum(t_a, t_b)

    t_lo = np.maximum(0.0, lo)
    t_hi = np.minimum(1.0, hi)

    valid = t_lo <= t_hi + eps

    return t_lo, t_hi, valid


def quadratic_value(Aq, Bq, Cq, t):

    return Aq * t * t + Bq * t + Cq


def quadratic_min_on_interval(Aq, Bq, Cq, t_lo, t_hi, valid, eps=EPS):

    best_t = t_lo.copy()
    best_q = quadratic_value(Aq, Bq, Cq, best_t)

    q_hi = quadratic_value(Aq, Bq, Cq, t_hi)
    use_hi = q_hi < best_q

    best_q[use_hi] = q_hi[use_hi]
    best_t[use_hi] = t_hi[use_hi]

    if Aq > eps:
        t_vertex = -Bq / (2.0 * Aq)

        use_vertex = (
            valid
            & (t_vertex >= t_lo - eps)
            & (t_vertex <= t_hi + eps)
        )

        if np.any(use_vertex):
            t_vertex_clipped = np.clip(t_vertex, t_lo, t_hi)
            q_vertex = quadratic_value(Aq, Bq, Cq, t_vertex_clipped)

            improve = use_vertex & (q_vertex < best_q)

            best_q[improve] = q_vertex[improve]
            best_t[improve] = t_vertex_clipped[improve]

    best_q[~valid] = np.inf
    best_t[~valid] = np.nan

    return best_q, best_t


def ctc_tip_touching_sphere_params(h3, h4, r3, r4, eps=EPS): # Spatial Hashing Broad Phase - Capsule Helpers

    h3 = float(h3)
    h4 = float(h4)
    r3 = float(r3)
    r4 = float(r4)

    H = h3 + h4

    terms = []
    terms.append((r3*r3 + h3*h3) / (2.0*h3))
    
    if h4 > eps:
        terms.append((r4*r4 + h3*h3) / (2.0*h3))
        terms.append((r4*r4 + H*H) / (2.0*H))

    k = max(terms)
    R = k

    return k, R


def choose_median_capsule_hash_cell_size(A, A_COLS, eps=EPS):

    radii = []

    for A_row_idx, row in enumerate(A):
        h3 = row[A_COLS["h3"]]
        h4 = row[A_COLS["h4"]]
        r3 = row[A_COLS["r3"]]
        r4 = row[A_COLS["r4"]]

        try:
            _, R = ctc_tip_touching_sphere_params(
                h3=h3,
                h4=h4,
                r3=r3,
                r4=r4,
                eps=eps,
            )

            if np.isfinite(R) and R > eps:
                radii.append(R)

        except SystemExit:
            pass

    if len(radii) == 0:
        raise SystemExit(
            "Could not choose cell size because no valid capsule radii were found."
        )

    radii = np.asarray(radii, dtype=np.float64)

    cell_size = float(np.median(radii))

    if not np.isfinite(cell_size) or cell_size <= eps:
        raise SystemExit("Chosen median cell size is invalid.")

    #print(f"Smallest capsule/sphere radius found: {np.min(radii):.6g}")
    #print(f"Median capsule/sphere radius used:    {cell_size:.6g}")
    #print(f"Largest capsule/sphere radius found:  {np.max(radii):.6g}")

    return cell_size


def point_cell_key(point_xyz, inv_cell_size): # Converts a point into a spatial hash cell key.

    return tuple(np.floor(point_xyz * inv_cell_size).astype(np.int64)) 


def add_point_to_spatial_hash(grid, point_xyz, point_local_idx, inv_cell_size): # Adds one accumulated A_ref point to the spatial hash

    key = point_cell_key(point_xyz, inv_cell_size)
    grid[key].append(int(point_local_idx))


def point_to_segment_distance_sq(points, a, b, eps=EPS):

    points = np.asarray(points, dtype=np.float64)
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)

    ab = b - a
    ab_sq = float(np.dot(ab, ab))

    if ab_sq <= eps:
        w = points - a
        return np.einsum("ij,ij->i", w, w)

    ap = points - a
    lam = (ap @ ab) / ab_sq
    lam = np.clip(lam, 0.0, 1.0)

    closest = a + lam[:, None] * ab
    diff = points - closest

    return np.einsum("ij,ij->i", diff, diff)


def capsule_hash_candidate_indices(points, grid, p0, p1, u, h3, h4, r3, r4, cell_size, eps=EPS):

    points = np.asarray(points, dtype=np.float64)

    if points.shape[0] == 0:
        return np.empty(0, dtype=np.int64), 0, 0, np.nan, np.nan

    inv_cell_size = 1.0 / cell_size

    p0 = np.asarray(p0, dtype=np.float64)
    p1 = np.asarray(p1, dtype=np.float64)
    u = np.asarray(u, dtype=np.float64)

    norm_u = np.linalg.norm(u)

    u = u / norm_u

    k_sphere, R_capsule = ctc_tip_touching_sphere_params(
        h3=h3,
        h4=h4,
        r3=r3,
        r4=r4,
        eps=eps,
    )

    # Swept bounding sphere centreline
    c0 = p0 + k_sphere * u
    c1 = p1 + k_sphere * u

    R_query = R_capsule + eps

    # AABB of the capsule
    bb_min = np.minimum(c0, c1) - R_query
    bb_max = np.maximum(c0, c1) + R_query

    cmin = np.floor(bb_min * inv_cell_size).astype(np.int64)
    cmax = np.floor(bb_max * inv_cell_size).astype(np.int64)

    candidate_lists = []

    for cx in range(cmin[0], cmax[0] + 1):
        for cy in range(cmin[1], cmax[1] + 1):
            for cz in range(cmin[2], cmax[2] + 1):

                ids = grid.get((cx, cy, cz))

                if ids:
                    candidate_lists.append(ids)

    if not candidate_lists:
        return np.empty(0, dtype=np.int64), 0, 0, k_sphere, R_capsule

    total_ids = sum(len(ids) for ids in candidate_lists)

    J = np.fromiter(
        (idx for ids in candidate_lists for idx in ids),
        dtype=np.int64,
        count=total_ids,
    )

    broad_cell_candidate_count = J.size

    if J.size == 0:
        return np.empty(0, dtype=np.int64), broad_cell_candidate_count, 0, k_sphere, R_capsule

    # Exact point-vs-capsule broad phase filter.
    Pj = points[J]

    dist_sq = point_to_segment_distance_sq(Pj, c0, c1, eps=eps)

    capsule_ok = dist_sq <= R_query * R_query

    J = J[capsule_ok]

    if J.size > 1:
        J = np.unique(J)

    capsule_candidate_count = J.size

    return J, broad_cell_candidate_count, capsule_candidate_count, k_sphere, R_capsule

In [8]:
# PIT Function

def swept_ctc_hits_points(points, p0, p1, u, h3, h4, r3, r4, h_0=0.0, tol=TOL, candidate_indices=None):

    points = np.asarray(points, dtype=float)
    n = points.shape[0]

    hit = np.zeros(n, dtype=bool)
    best_t = np.full(n, np.nan)
    best_axial = np.full(n, np.nan)
    best_radial = np.full(n, np.nan)
    best_allowed = np.full(n, np.nan)
    best_margin = np.full(n, -np.inf)
    best_section = np.full(n, "", dtype=object)

    empty_details = {
        "t": best_t,
        "axial": best_axial,
        "radial": best_radial,
        "allowed_radius": best_allowed,
        "margin": best_margin,
        "section": best_section,
    }

    if n == 0:
        return hit, empty_details

    p0 = np.asarray(p0, dtype=float)
    p1 = np.asarray(p1, dtype=float)
    u = np.asarray(u, dtype=float)

    norm_u = np.linalg.norm(u)

    u = u / norm_u

    h3 = float(h3)
    h4 = float(h4)
    r3 = float(r3)
    r4 = float(r4)
    h_0 = max(0.0, float(h_0))

    H = h3 + h4

    if H <= h_0 + EPS:
        return hit, empty_details

    # Candidate selection from broad phase
    if candidate_indices is None:
        idx = np.arange(n, dtype=np.int64)
        
    else:
        
        idx = np.asarray(candidate_indices, dtype=np.int64)
        idx = idx[(idx >= 0) & (idx < n)]
    
        if idx.size > 1:
            idx = np.unique(idx)
    
    if idx.size == 0:
        return hit, empty_details
    
    pts = points[idx]

    d = p1 - p0

    du = float(np.dot(d, u))
    d_perp = d - du * u
    d_perp_sq = float(np.dot(d_perp, d_perp))

    w = pts - p0

    a0 = w @ u
    w_perp = w - a0[:, None] * u

    w_perp_sq = np.einsum("ij,ij->i", w_perp, w_perp)
    w_perp_dot_d_perp = w_perp @ d_perp

    local_hit = np.zeros(len(idx), dtype=bool)
    local_t = np.full(len(idx), np.nan)
    local_axial = np.full(len(idx), np.nan)
    local_radial = np.full(len(idx), np.nan)
    local_allowed = np.full(len(idx), np.nan)
    local_margin = np.full(len(idx), -np.inf)
    local_section = np.full(len(idx), "", dtype=object)

    # CONE
    cone_s_min = h_0
    cone_s_max = min(h3, H)

    if h3 > EPS and r3 >= 0.0 and cone_s_max >= cone_s_min - EPS:
        k = r3 / h3

        t_lo, t_hi, valid = t_interval_for_axial_range(
            a0=a0,
            du=du,
            s_min=cone_s_min,
            s_max=cone_s_max,
        )

        Aq = d_perp_sq - (k * du) ** 2
        Bq = -2.0 * w_perp_dot_d_perp + 2.0 * (k ** 2) * a0 * du
        Cq = w_perp_sq - (k * a0) ** 2

        q_min, t_min = quadratic_min_on_interval(
            Aq=Aq,
            Bq=Bq,
            Cq=Cq,
            t_lo=t_lo,
            t_hi=t_hi,
            valid=valid,
        )

        cone_hit = q_min <= tol

        if np.any(cone_hit):
            axial = a0 - t_min * du

            radial_sq = (
                w_perp_sq
                - 2.0 * t_min * w_perp_dot_d_perp
                + (t_min ** 2) * d_perp_sq
            )

            radial = np.sqrt(np.maximum(0.0, radial_sq))
            allowed = k * axial
            margin = allowed - radial

            improve = cone_hit & (margin > local_margin)

            local_hit[improve] = True
            local_t[improve] = t_min[improve]
            local_axial[improve] = axial[improve]
            local_radial[improve] = radial[improve]
            local_allowed[improve] = allowed[improve]
            local_margin[improve] = margin[improve]
            local_section[improve] = "cone"

    # CYLINDER
    cyl_s_min = max(h_0, h3)
    cyl_s_max = H

    if h4 > EPS and r4 >= 0.0 and cyl_s_max >= cyl_s_min - EPS:
        t_lo, t_hi, valid = t_interval_for_axial_range(
            a0=a0,
            du=du,
            s_min=cyl_s_min,
            s_max=cyl_s_max,
        )

        Aq = d_perp_sq
        Bq = -2.0 * w_perp_dot_d_perp
        Cq = w_perp_sq

        q_min, t_min = quadratic_min_on_interval(
            Aq=Aq,
            Bq=Bq,
            Cq=Cq,
            t_lo=t_lo,
            t_hi=t_hi,
            valid=valid,
        )

        cyl_hit = q_min <= (r4 * r4 + tol)

        if np.any(cyl_hit):
            axial = a0 - t_min * du
            radial = np.sqrt(np.maximum(0.0, q_min))
            allowed = np.full_like(radial, r4)
            margin = allowed - radial

            improve = cyl_hit & (margin > local_margin)

            local_hit[improve] = True
            local_t[improve] = t_min[improve]
            local_axial[improve] = axial[improve]
            local_radial[improve] = radial[improve]
            local_allowed[improve] = allowed[improve]
            local_margin[improve] = margin[improve]
            local_section[improve] = "cyl"

    hit[idx] = local_hit
    best_t[idx] = local_t
    best_axial[idx] = local_axial
    best_radial[idx] = local_radial
    best_allowed[idx] = local_allowed
    best_margin[idx] = local_margin
    best_section[idx] = local_section

    details = {
        "t": best_t,
        "axial": best_axial,
        "radial": best_radial,
        "allowed_radius": best_allowed,
        "margin": best_margin,
        "section": best_section,
    }

    return hit, details

In [9]:
# Called Function - PIT Looper

def ccd_ctc_with_hash(h_0=h_0):
    
    t_start = perf_counter_ns()
    t_last = t_start

    def lap(name):
        nonlocal t_last
        now = perf_counter_ns()
        dt_ms = (now - t_last) / 1_000_000
        total_ms = (now - t_start) / 1_000_000
        print(f"{name:<45} /{dt_ms:10.3f}/ ms   total: {total_ms:10.3f} ms")
        t_last = now

    A, A_ref, A_COLS, A_REF_COLS = build_A_and_A_ref(h_0=h_0)

    lap("A and A_ref Build")

    cell_size = choose_median_capsule_hash_cell_size(A, A_COLS, eps=EPS)

    inv_cell_size = 1.0 / cell_size

    #print(f"Capsule spatial hash cell size: {cell_size:.6g}")

    # Maximum possible number of unique A_ref points is A_ref.shape[0]!
    test_point_xyz = np.empty((A_ref.shape[0], 3), dtype=np.float64)
    test_point_ref_rows = np.empty(A_ref.shape[0], dtype=np.int64)

    ref_row_added = np.zeros(A_ref.shape[0], dtype=bool)
    n_test_points = 0

    # Dynamic spatial hash of accumulated A_ref points
    # Each point is inserted once - no duplicates
    grid = defaultdict(list)

    hit_records = []
    skipped_bad_q1 = 0

    broad_cell_candidate_count = 0
    capsule_candidate_count = 0
    exact_test_count = 0

    max_capsule_candidates_one_row = 0
    max_broad_cell_candidates_one_row = 0

    lap("Spatial Hashing")

    loop_start = perf_counter_ns()

    for A_row_idx in range(A.shape[0]):
        row = A[A_row_idx]

        q1 = row[A_COLS["q1"]]
        q2 = row[A_COLS["q2"]]
        q3 = row[A_COLS["q3"]]

        ref_idx = q1_to_ref_index(q1)

        if ref_idx is None or ref_idx < 0 or ref_idx >= A_ref.shape[0]:
            skipped_bad_q1 += 1
            continue

        if not ref_row_added[ref_idx]:
            ref_row_added[ref_idx] = True

            new_point_xyz = A_ref[
                ref_idx,
                [A_REF_COLS["x"], A_REF_COLS["y"], A_REF_COLS["z"]]
            ].astype(np.float64)

            test_point_ref_rows[n_test_points] = ref_idx
            test_point_xyz[n_test_points, :] = new_point_xyz

            add_point_to_spatial_hash(
                grid=grid,
                point_xyz=new_point_xyz,
                point_local_idx=n_test_points,
                inv_cell_size=inv_cell_size,
            )

            n_test_points += 1

        points_now = test_point_xyz[:n_test_points]
        point_ref_rows_now = test_point_ref_rows[:n_test_points]

        p0 = row[[A_COLS["x"], A_COLS["y"], A_COLS["z"]]]
        p1 = row[[A_COLS["x+"], A_COLS["y+"], A_COLS["z+"]]]

        u = row[[A_COLS["i3"], A_COLS["j3"], A_COLS["k3"]]]

        h3 = row[A_COLS["h3"]]
        h4 = row[A_COLS["h4"]]
        r3 = row[A_COLS["r3"]]
        r4 = row[A_COLS["r4"]]

        # Capsule broad phase

        candidate_indices, broad_count, capsule_count, k_sphere, R_capsule = capsule_hash_candidate_indices(
            points=points_now,
            grid=grid,
            p0=p0,
            p1=p1,
            u=u,
            h3=h3,
            h4=h4,
            r3=r3,
            r4=r4,
            cell_size=cell_size,
            eps=EPS,
        )

        broad_cell_candidate_count += broad_count
        capsule_candidate_count += capsule_count
        exact_test_count += candidate_indices.size

        max_broad_cell_candidates_one_row = max(
            max_broad_cell_candidates_one_row,
            broad_count,
        )

        max_capsule_candidates_one_row = max(
            max_capsule_candidates_one_row,
            capsule_count,
        )

        if candidate_indices.size == 0:
            continue

        # PIT Test

        hit_mask, details = swept_ctc_hits_points(
            points=points_now,
            p0=p0,
            p1=p1,
            u=u,
            h3=h3,
            h4=h4,
            r3=r3,
            r4=r4,
            h_0=h_0,
            candidate_indices=candidate_indices,
        )

        hit_indices = np.flatnonzero(hit_mask)

        for local_point_idx in hit_indices:
            point_ref_row_zero_based = int(point_ref_rows_now[local_point_idx])
            point_xyz = points_now[local_point_idx]

            hit_records.append({
                "Sweep ID": A_row_idx,
                "Waypoint": point_ref_row_zero_based,

                "Q1": q1,
                "Q2": q2,
                "Q3": q3,

                "Point X": point_xyz[0],
                "Point Y": point_xyz[1],
                "Point Z": point_xyz[2],

                "Cyl/Cone": details["section"][local_point_idx],
                "Axial Tip Dist.": details["axial"][local_point_idx],
                "Radial Dist.": details["radial"][local_point_idx],
                "Allowed Radius": details["allowed_radius"][local_point_idx],
            })

    loop_ms = (perf_counter_ns() - loop_start) / 1_000_000
    
    lap("Broad Phase and Exact Tests")

    hit_pairs = pd.DataFrame(hit_records)

    lap("Hit List")

    print("-" * 75)
    print(f"{'TOTAL TIME:':<45} {(perf_counter_ns() - t_start) / 1_000_000:10.3f} ms")
    print(f"Rows in A:                         {A.shape[0]:,}")
    print(f"Rows in A_ref:                     {A_ref.shape[0]:,}")
    print(f"Unique A_ref points accumulated:   {n_test_points:,}")
    print(f"Spatial hash occupied cells:       {len(grid):,}")
    print(f"Broad cell candidates:             {broad_cell_candidate_count:,}")
    print(f"Capsule candidates:                {capsule_candidate_count:,}")
    print(f"Intersection tests:                {exact_test_count:,}")
    print(f"Hit pairs:                         {len(hit_pairs):,}")
    print(f"Cell size:                         {cell_size:.6g}")

    if skipped_bad_q1 > 0:
        print(f"Skipped rows because q1 could not be used as an A_ref row index: {skipped_bad_q1:,}")

    return hit_pairs

In [10]:
hit_pairs = ccd_ctc_with_hash()

hit_pairs

A and A_ref Build                             /   291.928/ ms   total:    291.928 ms
Spatial Hashing                               /    12.106/ ms   total:    304.034 ms
Broad Phase and Exact Tests                   /  2242.313/ ms   total:   2546.346 ms
Hit List                                      /     4.327/ ms   total:   2550.673 ms
---------------------------------------------------------------------------
TOTAL TIME:                                     2550.794 ms
Rows in A:                         1,995
Rows in A_ref:                     2,000
Unique A_ref points accumulated:   1,995
Spatial hash occupied cells:       11
Broad cell candidates:             1,729,933
Capsule candidates:                2,030
Intersection tests:                2,030
Hit pairs:                         0
Cell size:                         21.715


""
